# CS464 — ViT-B/16 Training
**Group 14** | George · Nana · Shaun · Victor

Self-contained notebook for training and evaluating the ViT-B/16 Vision Transformer on FaceForensics++ frames.

| Step | Script |
|------|--------|
| Model definition | `vit_model.py` |
| Training loop | `train_vit.py` |
| Data loading | `dataloaders.py` |

**Prerequisites:** frames already extracted to `src/data/train/`, `val/`, `test/`.

## 0. Environment Setup

In [1]:
import sys
import os
from pathlib import Path

# Add scripts folder to path so we can import from it
SCRIPTS_DIR = Path('../scripts').resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Key project paths
WORKSPACE_ROOT = Path('../..').resolve()
DATA_DIR       = WORKSPACE_ROOT / 'data'
MODELS_DIR     = WORKSPACE_ROOT / 'src' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Workspace : {WORKSPACE_ROOT}')
print(f'Data dir  : {DATA_DIR}  (exists: {DATA_DIR.exists()})')
print(f'Models dir: {MODELS_DIR}')

Workspace : /Users/vrasum/projects/deepfake-detection
Data dir  : /Users/vrasum/projects/deepfake-detection/data  (exists: True)
Models dir: /Users/vrasum/projects/deepfake-detection/src/models


In [2]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'MPS available   : {torch.backends.mps.is_available()}')

PyTorch version : 2.11.0
CUDA available  : False
MPS available   : True


## 1. Data Loading

In [3]:
from dataloaders import get_device, get_dataloaders, sanity_check

device  = get_device()
# ViT-B/16 is 86 M params — batch_size=16 is safe on 8.5 GB VRAM
# Increase to 32 if you have more VRAM available
loaders = get_dataloaders(DATA_DIR, batch_size=16, num_workers=4)

sanity_check(loaders)

[device] MPS — Apple Silicon GPU
DATALOADER SANITY CHECK

  [TRAIN]  73391 samples
    fake   (idx=0):    62906 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):    10485 samples   14.3%  ███████

  [VAL]  15750 samples
    fake   (idx=0):    13500 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):     2250 samples   14.3%  ███████

  [TEST]  15843 samples
    fake   (idx=0):    13578 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):     2265 samples   14.3%  ███████

  Class weights (for CrossEntropyLoss):
    fake   (idx=0):  weight = 0.5833
    real   (idx=1):  weight = 3.4998

  Fetching one sample batch from train loader...


/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


    images shape  : (16, 3, 224, 224)
    labels shape  : (16,)
    pixel range   : [-2.118, 2.640]
    batch fake    : 8
    batch real    : 8

Sanity check passed. Ready to train.


## 2. Model

ViT-B/16 pretrained on ImageNet, classification head replaced with `Linear(768 → 2)`.

| Property | Value |
|----------|-------|
| Patch size | 16×16 |
| Tokens | 196 patches + 1 CLS = 197 |
| Hidden dim | 768 |
| Transformer layers | 12 |
| Attention heads | 12 |
| Total parameters | ~86 M |

In [4]:
from vit_model import get_vit_model, freeze_backbone, unfreeze_backbone

vit = get_vit_model().to(device)

total_params = sum(p.numel() for p in vit.parameters())
trainable    = sum(p.numel() for p in vit.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable:,}')
print()
print(vit)

Total parameters     : 85,800,194
Trainable parameters : 85,800,194

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): 

## 3. Training (Two-Phase)

| Phase | Backbone | Epochs | LR | Purpose |
|-------|----------|--------|----|---------|
| 1 | Frozen | 10 | 1e-3 | Warm up classification head only |
| 2 | Unfrozen | 15 | 1e-5 | Full fine-tuning with tiny LR |

Checkpoints saved to `src/models/`. Use `vit_phase2_best.pth` for evaluation.

In [ ]:
from train_vit import train as train_vit

# start_phase=1  → run both phases (fresh start)
# start_phase=2  → skip Phase 1, load vit_phase1_best.pth and go straight to fine-tuning
train_vit(start_phase=1)

[device] MPS — Apple Silicon GPU
DATALOADER SANITY CHECK

  [TRAIN]  73391 samples
    fake   (idx=0):    62906 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):    10485 samples   14.3%  ███████

  [VAL]  15750 samples
    fake   (idx=0):    13500 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):     2250 samples   14.3%  ███████

  [TEST]  15843 samples
    fake   (idx=0):    13578 samples   85.7%  ██████████████████████████████████████████
    real   (idx=1):     2265 samples   14.3%  ███████

  Class weights (for CrossEntropyLoss):
    fake   (idx=0):  weight = 0.5833
    real   (idx=1):  weight = 3.4998

  Fetching one sample batch from train loader...


/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


    images shape  : (16, 3, 224, 224)
    labels shape  : (16,)
    pixel range   : [-2.118, 2.640]
    batch fake    : 9
    batch real    : 7

Sanity check passed. Ready to train.

Model: ViT-B/16  (85,800,194 total parameters)

[Phase 1] Backbone FROZEN — trainable: 1,538 / 85,800,194 parameters (0.0%)
Phase 1  |  epochs=10  lr=1e-03  batch=16


  Epoch  1/10 | train_loss=0.4084  train_acc=0.5781 | val_loss=1.4545  val_acc=0.2960  val_auc=0.3799 | lr=1.00e-03  (3301s)
  Checkpoint saved → vit_phase1_best.pth
  ✓ New best  (val_loss=1.4545  val_auc=0.3799)


  train:   0%|                                                                                                                                | 0/4586 [00:00<?, ?it/s]/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                                                                                                                                       

...

  train:   0%|                                                                                                                                | 0/4586 [00:00<?, ?it/s]/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                                                                                                                                       

  Epoch  3/10 | train_loss=0.3964  train_acc=0.5842 | val_loss=1.7018  val_acc=0.2879  val_auc=0.3680 | lr=1.00e-03  (5178s)


  train:   0%|                                                                                                                                | 0/4586 [00:00<?, ?it/s]/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                                                                                                                                       

  Epoch  4/10 | train_loss=0.3962  train_acc=0.5847 | val_loss=1.5721  val_acc=0.2964  val_auc=0.3690 | lr=1.00e-03  (4438s)


  train:   0%|                                                                                                             | 0/4586 [00:00<?, ?it/s]/Users/vrasum/projects/deepfake-detection/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  train:   2%|██▏                                                                                                 | 99/4586 [00:58<44:40,  1.67it/s]

## 4. Training Curves

In [ ]:
vit_log = pd.read_csv(MODELS_DIR / 'vit_training_log.csv')

phase_colors = {1: ('#4C72B0', '#DD8452'), 2: ('#55A868', '#C44E52')}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ph in sorted(vit_log['phase'].unique()):
    ph_log = vit_log[vit_log['phase'] == ph]
    offset = int(vit_log[vit_log['phase'] < ph]['epoch'].max()) if ph > 1 else 0
    ep     = ph_log['epoch'] + offset
    tc, vc = phase_colors[ph]

    axes[0].plot(ep, ph_log['train_loss'], color=tc, marker='o', markersize=3, label=f'Train P{ph}')
    axes[0].plot(ep, ph_log['val_loss'],   color=vc, marker='o', markersize=3, label=f'Val P{ph}', linestyle='--')

    axes[1].plot(ep, ph_log['train_acc'], color=tc, marker='o', markersize=3, label=f'Train P{ph}')
    axes[1].plot(ep, ph_log['val_acc'],   color=vc, marker='o', markersize=3, label=f'Val P{ph}', linestyle='--')

    axes[2].plot(ep, ph_log['val_auc'], color=vc, marker='o', markersize=3, label=f'Val AUC P{ph}')

# Phase boundary line
if len(vit_log['phase'].unique()) > 1:
    boundary = int(vit_log[vit_log['phase'] == 1]['epoch'].max())
    for ax in axes:
        ax.axvline(x=boundary, color='gray', linestyle=':', linewidth=1.2)
        ax.text(boundary + 0.1, ax.get_ylim()[0], 'Phase 2', fontsize=7, color='gray')

axes[0].set_title('Loss');        axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8)
axes[1].set_title('Accuracy');    axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)
axes[2].set_title('Val AUC-ROC'); axes[2].set_xlabel('Epoch'); axes[2].legend(fontsize=8)

plt.suptitle('ViT-B/16 — Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

print()
for ph in sorted(vit_log['phase'].unique()):
    ph_log = vit_log[vit_log['phase'] == ph]
    print(f"Phase {ph} — best val_auc : {ph_log['val_auc'].max():.4f}  "
          f"| best val_loss : {ph_log['val_loss'].min():.4f}  "
          f"| epochs run : {len(ph_log)}")

## 5. Checkpoint Info

In [ ]:
# Show saved checkpoints and their metadata
for ckpt_name in ['vit_phase1_best.pth', 'vit_phase2_best.pth', 'vit_final.pth']:
    ckpt_path = MODELS_DIR / ckpt_name
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location='cpu')
        info = {k: v for k, v in ckpt.items() if k != 'model_state_dict'}
        print(f"{ckpt_name}")
        for k, v in info.items():
            print(f"  {k}: {v}")
        print()
    else:
        print(f"{ckpt_name}  — not found (run training first)")